# Chapter 7 — Generating fake invoice PDFs

Move beyond a single fixture and produce a **realistic synthetic invoice corpus**.
Goal: test header **address detection** by exploiting the typographic convention
*"sender on the left, recipient on the right"*.

Real-world invoices are not uniform: some carry explicit `BILL TO` / `BILL FROM`
labels, others none ; some are letterheads, some minimalist ; titles range from
`INVOICE` to `STATEMENT` to nothing at all. The generator here covers a wide
spread of these structural variants so that downstream extractors can be
stress-tested against realistic noise.

## 0. Setup

In [ ]:
from __future__ import annotations

import os, sys, random
from pathlib import Path
from datetime import date, timedelta
from dataclasses import dataclass, field

if sys.platform == "win32":
    os.environ.setdefault("PYTHONUTF8", "1")
    try:
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    except AttributeError:
        pass

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT  = REPO / "notebooks" / "_outputs" / "invoices"
OUT.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)

print(f"Output : {OUT}")

for pkg in ("reportlab", "faker", "fitz"):
    try:
        __import__(pkg)
        print(f"  {pkg:12s} OK")
    except ImportError:
        print(f"  {pkg:12s} MISSING — pip install {pkg if pkg != 'fitz' else 'pymupdf'}")

## 1. The header convention

On US business invoices (and most Western templates), the typographic convention
places the **sender on the left** and the **recipient on the right** in the upper
third of the first page :

```
┌──────────────────────────────────────────┐
│ [LOGO]                                   │
│ <sender block>          <recipient block>│
│                                          │
│ INVOICE  INV-2024-0042                   │
│ ──────────────────────────────────────── │
│ Description           Qty  Rate  Amount  │
│ …                                        │
└──────────────────────────────────────────┘
```

The convention is **stable in roughly 85 % of US B2B invoices** even though many
details around it vary :

- Labels above each block range from explicit (`BILL FROM` / `BILL TO`) to short
  (`FROM` / `TO`) to nothing at all (especially on letterhead templates).
- The document title might be `INVOICE`, `STATEMENT`, `BILL`, or simply absent.
- Some invoices carry payment terms (`Net 30`), others a hard due date, or both.
- Logo presence and size vary widely.

What stays stable is the spatial layout : two text clusters in the upper band,
one on the left, one on the right. That is the signal a heuristic detector can
rely on, *before* any LLM/ML processing.

The remaining ~15 % of cases — sender on the right, centered headers, full-width
banners — are covered by the adversarial variants in §7.

## 2. Data models

In [ ]:
@dataclass
class Address:
    name:     str
    street:   str
    city:     str
    state:    str               # 2-letter US state
    zip_code: str
    ein:      str | None = None # Employer Identification Number (companies only)
    phone:    str | None = None
    email:    str | None = None

    def render_lines(self) -> list[str]:
        out = [self.street, f"{self.city}, {self.state} {self.zip_code}"]
        if self.phone:
            out.append(f"Tel: {self.phone}")
        if self.email:
            out.append(self.email)
        if self.ein:
            out.append(f"EIN: {self.ein}")
        return out


@dataclass
class LineItem:
    description: str
    quantity:    float
    unit_price:  float
    tax_rate:    float          # 0.0825 = 8.25 % sales tax

    @property
    def subtotal(self) -> float: return self.quantity * self.unit_price
    @property
    def tax(self) -> float:      return self.subtotal * self.tax_rate
    @property
    def total(self) -> float:    return self.subtotal + self.tax


@dataclass
class Invoice:
    number:        str
    issue_date:    date
    due_date:      date
    sender:        Address
    recipient:     Address
    items:         list[LineItem]
    notes:         str  = ""
    payment_terms: str  = ""             # "Net 30", "Due on receipt", or ""
    po_number:     str | None = None     # purchase-order reference

    @property
    def subtotal(self):  return sum(i.subtotal for i in self.items)
    @property
    def tax_total(self): return sum(i.tax      for i in self.items)
    @property
    def total(self):     return self.subtotal + self.tax_total

## 3. Fake data (Faker, US locale)

In [ ]:
from faker import Faker

fake = Faker("en_US")
Faker.seed(SEED)

_TAX_RATES = (0.0, 0.04, 0.0625, 0.0725, 0.0825, 0.10)
_PAYMENT_TERM_PHRASES = ("Net 15", "Net 30", "Net 30", "Net 45", "Net 60",
                         "Due on receipt", "", "")

_SERVICE_TEMPLATES = [
    "Strategic consulting",      "Software development",
    "Cloud infrastructure",      "Staff training",
    "Application maintenance",   "Annual software license",
    "Design services",           "Technical writing",
    "Security audit",            "Data migration",
    "Marketing retainer",        "Legal review",
    "Hourly engineering",        "Project management",
    "Onsite installation",       "Quarterly support",
]

_NOTE_TEMPLATES = [
    "Thank you for your business.",
    "Please remit payment to the address above. Late payments incur a 1.5 % monthly fee.",
    "Wire transfer instructions available upon request.",
    "Make checks payable to {sender_name}.",
    "Questions? Contact accounting@{sender_domain}.",
    "",   # no notes — common
    "",
]


def _random_ein() -> str:
    return f"{random.randint(10, 99)}-{random.randint(1000000, 9999999)}"


def random_address(*, is_company: bool = True) -> Address:
    if is_company:
        name = fake.company()
        ein  = _random_ein() if random.random() < 0.7 else None  # not always shown
    else:
        name = fake.name()
        ein  = None
    return Address(
        name=name,
        street=fake.street_address(),
        city=fake.city(),
        state=fake.state_abbr(),
        zip_code=fake.zipcode(),
        ein=ein,
        phone=fake.phone_number() if random.random() < 0.6 else None,
        email=fake.company_email() if (is_company and random.random() < 0.5) else None,
    )


def random_line_item() -> LineItem:
    desc = random.choice(_SERVICE_TEMPLATES)
    if random.random() < 0.6:
        desc += " — " + fake.bs()
    qty = random.choice([1, 1, 2, 4, 5, 8, 10, 12, 0.5, 1.5, 2.5])  # mix integer + fractional hours
    return LineItem(
        description=desc[:60],
        quantity=qty,
        unit_price=round(random.uniform(45, 1800), 2),
        tax_rate=random.choice(_TAX_RATES),
    )


def random_invoice(*, n_items: int | None = None) -> Invoice:
    issue = fake.date_between(start_date="-1y", end_date="today")
    due   = issue + timedelta(days=random.choice([15, 30, 30, 30, 45, 60]))
    items = [random_line_item() for _ in range(n_items or random.randint(1, 12))]
    sender = random_address(is_company=True)
    sender_domain = (sender.email or fake.domain_name()).split("@")[-1]
    note = random.choice(_NOTE_TEMPLATES).format(
        sender_name=sender.name, sender_domain=sender_domain,
    )
    return Invoice(
        number=f"INV-{issue.year}-{random.randint(1000, 9999)}",
        issue_date=issue,
        due_date=due,
        sender=sender,
        recipient=random_address(is_company=random.random() < 0.75),
        items=items,
        notes=note,
        payment_terms=random.choice(_PAYMENT_TERM_PHRASES),
        po_number=f"PO-{random.randint(10000, 99999)}" if random.random() < 0.4 else None,
    )


demo = random_invoice()
print(f"Number    : {demo.number}")
print(f"Sender    : {demo.sender.name}, {demo.sender.city}, {demo.sender.state}")
print(f"Recipient : {demo.recipient.name}, {demo.recipient.city}, {demo.recipient.state}")
print(f"Items     : {len(demo.items)}, total : ${demo.total:,.2f}")
print(f"Terms     : {demo.payment_terms or '(none)'}, PO: {demo.po_number or '(none)'}")
print(f"Notes     : {demo.notes[:60] or '(none)'}")

## 4. Style toggles

An `InvoiceStyle` dataclass groups all the structural variations the generator
knows about. The batch in §6 randomizes these per invoice.

In [ ]:
@dataclass
class InvoiceStyle:
    theme:        str   = "neutral"      # neutral | blue | green | purple | warm
    family:       str   = "Helvetica"    # Helvetica | Times-Roman
    logo_style:   str   = "box"          # box | banner | none
    sender_label: str   = "BILL FROM"    # "BILL FROM" | "FROM" | ""  (empty = no label)
    recip_label:  str   = "BILL TO"      # "BILL TO"   | "TO"   | "CUSTOMER" | ""
    title:        str   = "INVOICE"      # "INVOICE" | "STATEMENT" | "BILL" | ""
    show_po:      bool  = True
    show_terms:   bool  = True           # "Net 30" line under dates
    show_tax_col: bool  = True           # show TAX column in items table
    show_notes:   bool  = True           # footer notes
    swap_addresses: bool = False          # adversarial : sender on the right

## 5. PDF generator (reportlab)

In [ ]:
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import LETTER
from reportlab.lib.units import inch
from reportlab.lib import colors

_THEMES = {
    "neutral": {"primary": colors.HexColor("#1e293b"), "accent": colors.HexColor("#94a3b8")},
    "blue":    {"primary": colors.HexColor("#1d4ed8"), "accent": colors.HexColor("#dbeafe")},
    "green":   {"primary": colors.HexColor("#047857"), "accent": colors.HexColor("#d1fae5")},
    "purple":  {"primary": colors.HexColor("#6d28d9"), "accent": colors.HexColor("#ede9fe")},
    "warm":    {"primary": colors.HexColor("#b45309"), "accent": colors.HexColor("#fef3c7")},
}

_BOLD   = {"Helvetica": "Helvetica-Bold",    "Times-Roman": "Times-Bold"}
_ITALIC = {"Helvetica": "Helvetica-Oblique", "Times-Roman": "Times-Italic"}


def generate_invoice(
    invoice: Invoice,
    out_path: Path,
    style: InvoiceStyle | None = None,
) -> Path:
    """Render `invoice` to a PDF using the structural toggles in `style`."""
    style = style or InvoiceStyle()
    c = canvas.Canvas(str(out_path), pagesize=LETTER)
    W, H = LETTER
    palette = _THEMES.get(style.theme, _THEMES["neutral"])
    fam = style.family
    bold = _BOLD.get(fam, fam + "-Bold")

    sender, recipient = invoice.sender, invoice.recipient
    if style.swap_addresses:
        sender, recipient = recipient, sender

    x_left  = 0.75 * inch
    x_right = 4.75 * inch
    y_top   = H - 1.0 * inch

    # ── Logo ──────────────────────────────────────────────────────────
    if style.logo_style == "banner":
        c.setFillColor(palette["primary"])
        c.rect(0, H - 0.5 * inch, W, 0.5 * inch, fill=1, stroke=0)
        c.setFillColor(colors.white)
        c.setFont(bold, 16)
        c.drawString(x_left, H - 0.35 * inch, sender.name.upper())
        c.setFillColor(colors.black)
        sender_y = y_top - 0.35 * inch
    elif style.logo_style == "box":
        c.setFillColor(palette["accent"])
        c.rect(x_left, y_top - 0.7 * inch, 1.3 * inch, 0.7 * inch, fill=1, stroke=0)
        c.setFillColor(palette["primary"])
        c.setFont(bold, 14)
        c.drawString(x_left + 0.15 * inch, y_top - 0.45 * inch,
                     sender.name.split()[0][:10])
        c.setFillColor(colors.black)
        sender_y = y_top - 0.95 * inch
    else:  # "none"
        sender_y = y_top - 0.05 * inch

    # ── Sender block (left) ───────────────────────────────────────────
    cur_y = sender_y
    if style.sender_label:
        c.setFillColor(palette["accent"])
        c.setFont(bold, 9)
        c.drawString(x_left, cur_y, style.sender_label)
        cur_y -= 0.16 * inch
    c.setFillColor(palette["primary"])
    c.setFont(bold, 12)
    c.drawString(x_left, cur_y, sender.name)
    c.setFillColor(colors.black)
    c.setFont(fam, 9)
    for i, line in enumerate(sender.render_lines()):
        c.drawString(x_left, cur_y - (i + 1) * 0.16 * inch, line)

    # ── Recipient block (right) ───────────────────────────────────────
    cur_y = y_top
    if style.recip_label:
        c.setFillColor(palette["accent"])
        c.setFont(bold, 9)
        c.drawString(x_right, cur_y, style.recip_label)
        cur_y -= 0.16 * inch
    c.setFillColor(palette["primary"])
    c.setFont(bold, 12)
    c.drawString(x_right, cur_y, recipient.name)
    c.setFillColor(colors.black)
    c.setFont(fam, 9)
    for i, line in enumerate(recipient.render_lines()):
        c.drawString(x_right, cur_y - (i + 1) * 0.16 * inch, line)

    # ── Title + metadata ──────────────────────────────────────────────
    y_meta = H - 3.5 * inch
    if style.title:
        c.setFillColor(palette["primary"])
        c.setFont(bold, 22)
        c.drawString(x_left, y_meta, style.title)
        c.setFont(bold, 11)
        c.drawString(x_left + 1.7 * inch, y_meta + 0.04 * inch, invoice.number)
    else:
        c.setFillColor(palette["primary"])
        c.setFont(bold, 14)
        c.drawString(x_left, y_meta, f"# {invoice.number}")
    c.setFillColor(colors.black)
    c.setFont(fam, 10)
    c.drawString(x_left, y_meta - 0.30 * inch,
                 f"Issue date : {invoice.issue_date.strftime('%m/%d/%Y')}")
    c.drawString(x_left, y_meta - 0.50 * inch,
                 f"Due date   : {invoice.due_date.strftime('%m/%d/%Y')}")
    extra_y = y_meta - 0.70 * inch
    if style.show_terms and invoice.payment_terms:
        c.drawString(x_left, extra_y, f"Terms      : {invoice.payment_terms}")
        extra_y -= 0.20 * inch
    if style.show_po and invoice.po_number:
        c.drawString(x_left, extra_y, f"PO         : {invoice.po_number}")

    # ── Items table ───────────────────────────────────────────────────
    y_tbl = H - 5.2 * inch
    c.setFillColor(palette["primary"])
    c.rect(x_left - 0.05 * inch, y_tbl - 0.05 * inch, 7.0 * inch, 0.25 * inch,
           fill=1, stroke=0)
    c.setFillColor(colors.white)
    c.setFont(bold, 9)
    c.drawString     (x_left,             y_tbl + 0.05 * inch, "DESCRIPTION")
    c.drawRightString(x_left + 4.2 * inch, y_tbl + 0.05 * inch, "QTY")
    c.drawRightString(x_left + 5.2 * inch, y_tbl + 0.05 * inch, "RATE")
    if style.show_tax_col:
        c.drawRightString(x_left + 5.9 * inch, y_tbl + 0.05 * inch, "TAX")
    c.drawRightString(x_left + 6.9 * inch, y_tbl + 0.05 * inch, "AMOUNT")

    c.setFillColor(colors.black)
    c.setFont(fam, 9)
    y = y_tbl - 0.25 * inch
    for item in invoice.items:
        c.drawString     (x_left,             y, item.description[:55])
        qty_str = f"{item.quantity:g}"   # drops trailing zeros (1.5 not 1.50)
        c.drawRightString(x_left + 4.2 * inch, y, qty_str)
        c.drawRightString(x_left + 5.2 * inch, y, f"${item.unit_price:,.2f}")
        if style.show_tax_col:
            c.drawRightString(x_left + 5.9 * inch, y, f"{item.tax_rate * 100:.2f}%")
        c.drawRightString(x_left + 6.9 * inch, y, f"${item.subtotal:,.2f}")
        y -= 0.20 * inch

    # ── Totals ────────────────────────────────────────────────────────
    y -= 0.15 * inch
    c.line(x_left + 4.5 * inch, y, x_left + 6.9 * inch, y)
    y -= 0.25 * inch
    c.setFont(fam, 10)
    c.drawRightString(x_left + 5.7 * inch, y, "Subtotal:")
    c.drawRightString(x_left + 6.9 * inch, y, f"${invoice.subtotal:,.2f}")
    if invoice.tax_total > 0:
        c.drawRightString(x_left + 5.7 * inch, y - 0.20 * inch, "Tax:")
        c.drawRightString(x_left + 6.9 * inch, y - 0.20 * inch, f"${invoice.tax_total:,.2f}")
        y -= 0.20 * inch
    c.setFillColor(palette["primary"])
    c.setFont(bold, 12)
    c.drawRightString(x_left + 5.7 * inch, y - 0.30 * inch, "TOTAL DUE:")
    c.drawRightString(x_left + 6.9 * inch, y - 0.30 * inch, f"${invoice.total:,.2f}")

    # ── Notes / footer ────────────────────────────────────────────────
    if style.show_notes and invoice.notes:
        c.setFillColor(colors.black)
        c.setFont(_ITALIC.get(fam, fam + "-Oblique"), 9)
        c.drawString(x_left, 1.0 * inch, invoice.notes[:90])

    c.setFillColor(palette["accent"])
    c.setFont(fam, 7)
    footer = sender.name
    if sender.ein:
        footer += f"  ·  EIN {sender.ein}"
    footer += f"  ·  Invoice {invoice.number}"
    c.drawString(x_left, 0.5 * inch, footer)

    c.save()
    return out_path


demo_path = generate_invoice(demo, OUT / "_demo.pdf",
                              InvoiceStyle(theme="blue", logo_style="box"))
print(f"Demo written : {demo_path.relative_to(REPO)}  ({demo_path.stat().st_size // 1024} KB)")

## 6. Batch generation — 16 invoices, real-world variety

Each invoice picks **independent** values for theme, font, logo style, label
phrasings, title, and which optional fields are shown — to mimic the spread of
templates a downstream extractor will encounter.

In [ ]:
_THEME_OPTS  = ("neutral", "blue", "green", "purple", "warm")
_LOGO_OPTS   = ("box", "box", "banner", "none")            # "box" is the most common
_SENDER_LBL  = ("BILL FROM", "FROM", "", "", "")           # empty = letterhead style
_RECIP_LBL   = ("BILL TO", "TO", "CUSTOMER", "INVOICE TO", "")  # sometimes none
_TITLE_OPTS  = ("INVOICE", "INVOICE", "INVOICE", "STATEMENT", "BILL", "")
_FAMILY_OPTS = ("Helvetica", "Helvetica", "Times-Roman")    # Helvetica more frequent


def random_style() -> InvoiceStyle:
    return InvoiceStyle(
        theme        = random.choice(_THEME_OPTS),
        family       = random.choice(_FAMILY_OPTS),
        logo_style   = random.choice(_LOGO_OPTS),
        sender_label = random.choice(_SENDER_LBL),
        recip_label  = random.choice(_RECIP_LBL),
        title        = random.choice(_TITLE_OPTS),
        show_po      = random.random() < 0.7,
        show_terms   = random.random() < 0.6,
        show_tax_col = random.random() < 0.8,
        show_notes   = random.random() < 0.6,
        swap_addresses = False,
    )


random.seed(SEED)
Faker.seed(SEED)

N_INVOICES = 16
generated = []
for i in range(1, N_INVOICES + 1):
    inv   = random_invoice()
    style = random_style()
    out   = OUT / f"invoice_{i:02d}.pdf"
    generate_invoice(inv, out, style)
    generated.append((out, inv, style))

print(f"{N_INVOICES} invoices written. Style spread :\n")
for out, inv, st in generated:
    s_lbl = st.sender_label or "·"
    r_lbl = st.recip_label  or "·"
    title = st.title        or "·"
    print(f"  {out.name}  {st.theme:8s} {st.family:11s} logo={st.logo_style:6s} "
          f"title={title:9s} from={s_lbl:9s} to={r_lbl:10s} ${inv.total:>9,.2f}")

## 7. Visual validation of address bounding boxes

Heuristic : on the upper third of page 1, identify text blocks, split on the
midline. Leftmost cluster = sender ; rightmost cluster = recipient. Pure geometry,
no LLM.

In [ ]:
import fitz

def detect_address_zones(
    pdf_path: Path,
    *,
    upper_band: float = 0.40,
    midline:    float = 0.50,
) -> tuple[fitz.Rect | None, fitz.Rect | None]:
    with fitz.open(pdf_path) as doc:
        page = doc[0]
        W, H = page.rect.width, page.rect.height
        upper_y = H * upper_band
        mid_x   = W * midline
        blocks = page.get_text("blocks")
        in_band = [b for b in blocks if b[1] < upper_y and b[4].strip()]
        left_blocks  = [b for b in in_band if (b[0] + b[2]) / 2 <  mid_x]
        right_blocks = [b for b in in_band if (b[0] + b[2]) / 2 >= mid_x]

    def _hull(blocks):
        if not blocks:
            return None
        return fitz.Rect(
            min(b[0] for b in blocks), min(b[1] for b in blocks),
            max(b[2] for b in blocks), max(b[3] for b in blocks),
        )
    return _hull(left_blocks), _hull(right_blocks)


def render_with_bboxes(pdf_path: Path, png_path: Path, dpi: int = 144) -> Path:
    sender_bbox, recipient_bbox = detect_address_zones(pdf_path)
    doc = fitz.open(pdf_path)
    page = doc[0]
    if sender_bbox:
        page.draw_rect(sender_bbox, color=(0.05, 0.6, 0.2), width=1.8)
        page.insert_text(sender_bbox.tl + fitz.Point(0, -3),
                          "sender (left)", fontsize=8, color=(0.05, 0.6, 0.2))
    if recipient_bbox:
        page.draw_rect(recipient_bbox, color=(0.1, 0.3, 0.8), width=1.8)
        page.insert_text(recipient_bbox.tl + fitz.Point(0, -3),
                          "recipient (right)", fontsize=8, color=(0.1, 0.3, 0.8))
    pix = page.get_pixmap(matrix=fitz.Matrix(dpi / 72, dpi / 72))
    pix.save(str(png_path))
    doc.close()
    return png_path


PNG_DIR = OUT / "_annotated"
PNG_DIR.mkdir(exist_ok=True)

for out, _, _ in generated:
    png = PNG_DIR / (out.stem + "_annotated.png")
    render_with_bboxes(out, png)

print(f"Annotated PNGs in {PNG_DIR.relative_to(REPO)}/  ({len(generated)} files)")

Inline preview of one annotated invoice :

In [ ]:
from IPython.display import Image, display

sample_png = PNG_DIR / "invoice_03_annotated.png"
display(Image(str(sample_png), width=600))

## 8. Adversarial variants

Variants that **break** the convention, to verify that a future extractor handles
the long tail :

- `swap_addresses=True` — sender on the right (uncommon templates, scans from
  older systems, certain industries)
- no logo, no labels — pure minimalist
- banner header that pushes the address blocks down significantly

If the "left = sender" heuristic fails to ~50 % on these, that is the expected
result : these are the ~15 % of cases where you need a higher layer (NER,
classifier, or user confirmation).

In [ ]:
ADV_DIR = OUT / "_adversarial"
ADV_DIR.mkdir(exist_ok=True)

adversarial_styles = [
    InvoiceStyle(theme="neutral", logo_style="none",   sender_label="",          recip_label="",         title="",        swap_addresses=True),
    InvoiceStyle(theme="blue",    logo_style="banner", sender_label="BILL FROM", recip_label="BILL TO",  title="INVOICE", swap_addresses=True),
    InvoiceStyle(theme="warm",    logo_style="box",    sender_label="FROM",      recip_label="TO",       title="BILL",     swap_addresses=True),
    InvoiceStyle(theme="purple",  logo_style="banner", sender_label="",          recip_label="",         title="STATEMENT", swap_addresses=False),  # control
]

random.seed(SEED + 100)
Faker.seed(SEED + 100)

for i, st in enumerate(adversarial_styles, start=1):
    inv = random_invoice()
    out = ADV_DIR / f"adv_{i:02d}.pdf"
    generate_invoice(inv, out, st)
    png = ADV_DIR / (out.stem + "_annotated.png")
    render_with_bboxes(out, png)
    flag = "⚠ swapped" if st.swap_addresses else "   normal "
    print(f"  {flag}  {out.name}  true sender : {inv.sender.name[:30]}")

print(f"\n→ {len(adversarial_styles)} adversarial invoices in {ADV_DIR.relative_to(REPO)}/")

## 9. Recap

Outputs :

- `notebooks/_outputs/invoices/invoice_*.pdf` — 16 standard invoices (mixed labels,
  themes, logos, titles, optional fields)
- `notebooks/_outputs/invoices/_annotated/*.png` — same as PNG, with bbox overlays
- `notebooks/_outputs/invoices/_adversarial/` — variants that violate the convention

Reusable building blocks :

- `Address`, `LineItem`, `Invoice` — data models
- `InvoiceStyle` — structural toggles (label phrasings, logo style, optional fields)
- `random_invoice()` / `random_style()` — Faker en_US generators
- `generate_invoice(inv, out_path, style)` — reportlab PDF rendering
- `detect_address_zones(pdf)` — "upper band + midline" heuristic returning `(sender_bbox, recipient_bbox)`
- `render_with_bboxes(pdf, png)` — annotated visualization

**Natural next step** (notebook 08 ?) : extract the text inside each bbox via
`fitz.Page.get_text("text", clip=bbox)`, then parse into structured fields
(name / street / city / state / zip / EIN / phone / email). Sender / recipient
separation is already handled by the convention, so what remains is purely a
field-level structural problem.